This notebook aims to check if the timestamps are shifted for daylight savings time ( goes from CST to CDT); and if so, adjust the times so all timestamps are in CST.  

# Daylight Savings Time (DST)

- DST starts on the second sunday of March and ends on the first sunday of november. 

- The DST indicators: 

    - When DST starts, clocks will jump forward an hour at 2am to 3am. 

    - When DST ends, clocks will jump back an hour from 2am to 1am

# Checking for DST

To check for DST we first need the days in which it starts and ends

In [2]:
import pandas as pd

def second_sunday_of_march(year):
    """ Returns the date of the second Sunday of March for a given year."""
    first = pd.Timestamp(year, 3, 1)
    # days until Sunday (weekday 6)
    days_to_sun = (6 - first.weekday()) # if "first" is Monday then first.weekday() is 0, sunday is 6 
    return first + pd.Timedelta(days=days_to_sun + 7)  # +7 for second Sunday

def first_sunday_of_november(year):
    """ Returns the date of the first Sunday of November for a given year."""
    first = pd.Timestamp(year, 11, 1)
    days_to_sun = (6 - first.weekday())
    return first + pd.Timedelta(days=days_to_sun)

year = 2014 # I manually checked 2014 - 2024 and it is correct for all of them
second_sunday = second_sunday_of_march(year)
first_sunday = first_sunday_of_november(year)
print(f"Second Sunday of March {year}: {second_sunday}")
print(f"First Sunday of November {year}: {first_sunday}")

Second Sunday of March 2014: 2014-03-09 00:00:00
First Sunday of November 2014: 2014-11-02 00:00:00


Now that we have the days that DST start and end, lets find them in the data.

In [3]:
abs_data_path = r'C:\pyt\tx-soil-moisture\datasets\TxSON_data_2026-02-24\CB25.dat'
soil = pd.read_csv(abs_data_path, header = 5, parse_dates=['Date'], index_col='Date')

In [3]:
# Check for the DST transition days in the soil moisture data
def transition_day_data(year, data, day_offset = 0):
    dst_start = second_sunday_of_march(year) + pd.Timedelta(days=day_offset)  
    dst_end = first_sunday_of_november(year) + pd.Timedelta(days=day_offset)
    
    # get the data from midnight to 4 am on the transition days as one whole DataFrame
    start_data = data[dst_start:dst_start + pd.Timedelta(hours=4)]
    end_data = data[dst_end:dst_end + pd.Timedelta(hours=4)]

    return start_data, end_data

In [4]:
def missing_two_am_DST(data, year, day_offset=0):
    """
    1. Checks if there is a missing 2 AM entry on the day of DST start or a certain day after.
    2. Also make sure the surrounding entries are present to confirm it's not just a missing timestamp but an actual DST transition.
    """
    dst_start = second_sunday_of_march(year) + pd.Timedelta(days=day_offset, hours=2)  # 2 AM on or after the day DST starts

    if ( (dst_start - pd.Timedelta(hours=1)) not in data.index) and ( (dst_start + pd.Timedelta(hours=1)) not in data.index ):
        # If both 1 AM and 3 AM entries are missing, we can't confirm it's a DST issue
        return False 
    
    return dst_start not in data.index

In [5]:
def duplicate_one_am_DST(data, year, day_offset=0, diff_measurements=False):
    """Check for duplicate 1 AM entries (with or without identical measurements)."""
    
    dst_end = first_sunday_of_november(year) + pd.Timedelta(days=day_offset, hours=1) # 1 AM on or after the day DST ends

    dup = data.loc[data.index == dst_end]   # boolean, preserves Series/DataFrame type

    if len(dup) <= 1:
        return False                         # no duplicate timestamp at all

    if diff_measurements:
        if isinstance(dup, pd.DataFrame):    # DataFrame case
            return dup.drop_duplicates().shape[0] > 1 # if there are more than 1 unique rows, then they are different measurements
        return dup.nunique() > 1             # Series case, if there are more than 1 unique values, then they are different measurements
    return True

In [7]:
abs_data_path = r'C:\pyt\tx-soil-moisture\datasets\TxSON_data_2026-02-24\CB25.dat'
soil = pd.read_csv(abs_data_path, header = 5, parse_dates=['Date'], index_col='Date')

year = 2015

start_data, end_data = transition_day_data(year, soil)
print("Data for the day DST starts:")
print(start_data)
print("\nDoes the data have a missing 2 AM entry on the DST start day?", missing_two_am_DST(soil, year))
print("\nData for the day DST ends:")
print(end_data)
print("\nDoes the data have a duplicate 1 AM entry on the DST end day?", duplicate_one_am_DST(soil, year))

Data for the day DST starts:
                      Ppt  SWC_5  SWC_10  SWC_20  SWC_50    T_5   T_10   T_20  \
Date                                                                            
2015-03-08 00:00:00  0.00  0.331   0.368   0.390   0.418  11.18  11.78  11.35   
2015-03-08 01:00:00  0.25  0.331   0.368   0.390   0.418  10.69  11.53  11.31   
2015-03-08 02:00:00  0.25  0.331   0.368   0.390   0.418  10.31  11.24  11.27   
2015-03-08 03:00:00  0.51  0.331   0.367   0.390   0.418  10.25  11.02  11.20   
2015-03-08 04:00:00  0.76  0.332   0.367   0.389   0.418  10.13  10.87  11.13   

                      T_50  Flag  
Date                              
2015-03-08 00:00:00  10.70     0  
2015-03-08 01:00:00  10.73     0  
2015-03-08 02:00:00  10.76     0  
2015-03-08 03:00:00  10.80     0  
2015-03-08 04:00:00  10.84     0  

Does the data have a missing 2 AM entry on the DST start day? False

Data for the day DST ends:
                     Ppt  SWC_5  SWC_10  SWC_20  SWC_50    T_

**There are no signs of the network observing DST.** 

There is no gap from 1am to 3am in march.

There is no duplicate timestamp at 1am in november 

# The Problem

If we look just a day after the beginning and ending of DST we see a duplicate 1 am timestamp with different measurements, the day after DST ends.

In [8]:
day_offset = 1
year = 2015

abs_data_path = r'C:\pyt\tx-soil-moisture\datasets\TxSON_data_2026-02-24\CB25.dat'
soil = pd.read_csv(abs_data_path, header = 5, parse_dates=['Date'], index_col='Date')

start_data, end_data = transition_day_data(year, soil, day_offset = day_offset) # check the day after the transition days
print("Data for the day after DST starts:")
print(start_data)
print("\nDoes the data have a missing 2 AM entry on the day after DST starts?", missing_two_am_DST(soil, year, day_offset = day_offset))
print("\nData for the day after DST ends:")
print(end_data)
print("\nDoes the data have a duplicate 1 AM entry on the day after DST ends?", duplicate_one_am_DST(soil, year, day_offset = day_offset))

Data for the day after DST starts:
                      Ppt  SWC_5  SWC_10  SWC_20  SWC_50    T_5   T_10   T_20  \
Date                                                                            
2015-03-09 00:00:00  0.00  0.356   0.374   0.389   0.419  11.84  11.83  11.35   
2015-03-09 01:00:00  0.00  0.356   0.374   0.389   0.419  11.74  11.78  11.36   
2015-03-09 02:00:00  0.25  0.357   0.374   0.390   0.420  11.65  11.72  11.39   
2015-03-09 03:00:00  0.00  0.357   0.375   0.390   0.420  11.54  11.68  11.40   
2015-03-09 04:00:00  0.00  0.358   0.375   0.390   0.420  11.41  11.61  11.38   

                      T_50  Flag  
Date                              
2015-03-09 00:00:00  11.18     0  
2015-03-09 01:00:00  11.21     0  
2015-03-09 02:00:00  11.22     0  
2015-03-09 03:00:00  11.23     0  
2015-03-09 04:00:00  11.25     0  

Does the data have a missing 2 AM entry on the day after DST starts? False

Data for the day after DST ends:
                     Ppt  SWC_5  SWC_10  S

The day after dst starts, there is STILL NO INDICATION. 

The day after dst ends, there is **NOW A INDICATION**. 

# Checking All Stations

lets count how many of the stations have a duplicate 1am timestamp the day after dst ends in a given year.

To go through the folder, we first must distinguish between met, soil, and extraneous/unknown files so that we can read them in properly. 

In [6]:
import os
from soil_or_met import SoilOrMet

folder = r'C:\pyt\tx-soil-moisture\datasets\TxSON_data_2026-02-24'

expected_soil_header = "Date,Ppt,SWC_5,SWC_10,SWC_20,SWC_50,T_5,T_10,T_20,T_50,Flag"
expected_met_header = "TIMESTAMP,RECORD,Rain_mm_Tot,AirTC_Avg,RH,WS_ms_S_WVT,WindDir_D1_WVT,SlrW_Avg,ETos,Rso"

# initialize the class
soil_or_met = SoilOrMet(expected_met_header, expected_soil_header, min_soil_features=9, min_met_features=10) # this min_soil_features allows all soil files to be classified

def get_file_paths(folder): 
    soil, met, unknown = [], [], []
    for file in os.listdir(folder):
        
        file_path = os.path.join(folder, file)

        if os.path.isfile(file_path):
            data_type = soil_or_met.determine_data_file(file_path)
            if data_type == 'soil':
                soil.append(file_path) # add the file path to the soil list
            elif data_type == 'met':
                met.append(file_path)
            else:
                unknown.append(file_path)
    return soil, met, unknown

In [10]:
soil_paths, met_paths, unknown_paths = get_file_paths(folder)

print(f"Soil files: {len(soil_paths)}")
print(f"Met files: {len(met_paths)}")
print(f"Unknown files: {len(unknown_paths)}")

Soil files: 33
Met files: 6
Unknown files: 2


Now we have the file paths to the soil and met files. We can now read them in properly and check for the dst indicators.

In [7]:
def read_soil(file):
    df = pd.read_csv(file, header=5, dtype = str)
    # convert Date to datetime and set as index
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
    # drop flag
    df = df.drop(columns=['Flag'])
    # drop rows with full row duplicates
    # df = df.drop_duplicates() # THIS NEEDS TO BE HANDLED BETTER. It is possible that two 1am have the same measurements.
    # set Date as index
    df = df.set_index('Date')
    # sort by index just in case
    df = df.sort_index() # THERE ARE SOME FILES WITH OUT-OF-ORDER DATES.

    return df

def read_met(file):
    df = pd.read_csv(file, header = 6, dtype = str)
    df = df.drop(df.index[0:2])
    df = df.drop(columns=['RECORD'])
    df['TIMESTAMP'] = pd.to_datetime(df['TIMESTAMP'], errors='coerce')
    # df = df.drop_duplicates() # THIS NEEDS TO BE HANDLED BETTER. It is possible that two 1am have the same measurements.
    df = df.set_index("TIMESTAMP")
    df = df.sort_index() # THERE ARE SOME FILES WITH OUT-OF-ORDER DATES.

    return df

In [8]:
# first read in all soil files and met files into dictionaries to avoid reading the same file multiple times
def read_all_files_into_dict(file_paths, read_function):
    data_dict = {}
    for file in file_paths:
        try:
            df = read_function(file)
        except:
            print(f"Error reading {file}. Skipping.")
            continue

        data_dict[file] = df

    return data_dict

In [9]:
def search_stations_for_dst_indicators(soil_files_dict, met_files_dict, year, day_offset = 0, diff_measurements=False):

    file_with_duplicate_one_am = []
    file_with_missing_two_am = []

    for file in soil_files_dict.keys():
        try:
            df = soil_files_dict[file]
        except:
            print(f"Error reading {file}. Skipping.")
            continue

        if duplicate_one_am_DST(df, year, day_offset, diff_measurements=diff_measurements):
            file_with_duplicate_one_am.append(file)
        if missing_two_am_DST(df, year, day_offset):
            file_with_missing_two_am.append(file)


    for file in met_files_dict.keys():
        try:
            df = met_files_dict[file]
        except:
            print(f"Error reading {file}. Skipping.")
            continue

        if duplicate_one_am_DST(df, year, day_offset, diff_measurements=diff_measurements):
            file_with_duplicate_one_am.append(file)
        if missing_two_am_DST(df, year, day_offset):
            file_with_missing_two_am.append(file)

    return file_with_duplicate_one_am, file_with_missing_two_am


In [10]:
def print_dst_full_summary(soil_files_dict, met_files_dict, year, diff_measurements=False):

    # print the year we are checking in green
    print(f"\033[32m\nChecking for DST indicators in {year}\033[0m")
    
    dst_end_ind, dst_start_ind = search_stations_for_dst_indicators(soil_files_dict, met_files_dict, year, day_offset = 0, diff_measurements=diff_measurements)
    dst_end_ind_offset_one, dst_start_ind_offset_one = search_stations_for_dst_indicators(soil_files_dict, met_files_dict, year, day_offset = 1, diff_measurements=diff_measurements)

    # if any are not empty, then print the counts
    if dst_end_ind or dst_start_ind or dst_end_ind_offset_one or dst_start_ind_offset_one:

        ### On the days ###

        print("On the day of the DST transitions:")
        print(f"\tFiles with the missing 2 AM entries: {len(dst_start_ind)}")
        print(f"\tFiles with the duplicate 1 AM entries: {len(dst_end_ind)}")
        # if there are indictions of dst on the day, print the files
        if dst_end_ind or dst_start_ind:
            print(f"\tFile paths of the files with DST indicators:")
            for file in dst_end_ind:
                print(f"\t\tDuplicate 1 AM entry: {file}")
            for file in dst_start_ind:
                print(f"\t\tMissing 2 AM entry: {file}")
        # if there is a missing 2am print the file
        if dst_start_ind:
            print(f"\tFile paths of the missing 2 AM entries:")
            for file in dst_start_ind:
                print(f"\t\t{file}")
        # print files that are in both dst_end_ind and dst_start_ind
        if dst_end_ind and dst_start_ind:
            common_files = set(dst_end_ind) & set(dst_start_ind)
            print(f"\tFiles with the duplicate 1 AM entries AND the missing 2 AM entries ON the days of DST transitions: {common_files if common_files else 'None'}")

        ### Right after the days ###

        print("One day AFTER the day of the DST transitions:")
        print(f"\tFiles with the duplicate 1 AM entries: {len(dst_end_ind_offset_one)}")
        print(f"\tFiles with the missing 2 AM entries: {len(dst_start_ind_offset_one)}")
        if dst_start_ind_offset_one:
            print(f"\tFile Paths of the files with the duplicate 2 AM entries:")
            for file in dst_start_ind_offset_one:
                print(f"\t\t{file}")
        # print files that are in both dst_end_ind_offset_one and dst_start_ind_offset_one
        if dst_end_ind_offset_one and dst_start_ind_offset_one:
            common_files_offset_one = set(dst_end_ind_offset_one) & set(dst_start_ind_offset_one)
            print(f"\tFiles with the duplicate 1 AM entries AND the missing 2 AM entries: {common_files_offset_one if common_files_offset_one else 'None'}")

    else:
        print(f"\tNo DST indicators found in {year}.")

In [15]:
soil_files_dict = read_all_files_into_dict(soil_paths, read_soil)
met_files_dict = read_all_files_into_dict(met_paths, read_met)
print_dst_full_summary(soil_files_dict, met_files_dict, 2017, diff_measurements=False)


Checking for DST indicators in 2017
On the day of the DST transitions:
	Files with the missing 2 AM entries: 0
	Files with the duplicate 1 AM entries: 2
	File paths of the files with DST indicators:
		Duplicate 1 AM entry: C:\pyt\tx-soil-moisture\datasets\TxSON_data_2026-02-24\CB06.dat
		Duplicate 1 AM entry: C:\pyt\tx-soil-moisture\datasets\TxSON_data_2026-02-24\FD03_met.dat
One day AFTER the day of the DST transitions:
	Files with the duplicate 1 AM entries: 2
	Files with the missing 2 AM entries: 0


In [16]:
print_dst_full_summary(soil_files_dict, met_files_dict, 2017, diff_measurements=True) # requires diff measurements for the dup 1 am check (indicator of dst end).


Checking for DST indicators in 2017


	No DST indicators found in 2017.


Now lets put it all together and report the number of files with duplicate 1am timestamps, the day after dst ends, by year.

In [17]:
# soil_files_dict = read_all_files_into_dict(soil_paths, read_soil)
# met_files_dict = read_all_files_into_dict(met_paths, read_met)

for year in range(2014, 2025):
    print_dst_full_summary(soil_files_dict, met_files_dict, year)


Checking for DST indicators in 2014
	No DST indicators found in 2014.

Checking for DST indicators in 2015
On the day of the DST transitions:
	Files with the missing 2 AM entries: 0
	Files with the duplicate 1 AM entries: 0
One day AFTER the day of the DST transitions:
	Files with the duplicate 1 AM entries: 22
	Files with the missing 2 AM entries: 0

Checking for DST indicators in 2016
On the day of the DST transitions:
	Files with the missing 2 AM entries: 0
	Files with the duplicate 1 AM entries: 2
	File paths of the files with DST indicators:
		Duplicate 1 AM entry: C:\pyt\tx-soil-moisture\datasets\TxSON_data_2026-02-24\FD12.dat
		Duplicate 1 AM entry: C:\pyt\tx-soil-moisture\datasets\TxSON_data_2026-02-24\WC05.dat
One day AFTER the day of the DST transitions:
	Files with the duplicate 1 AM entries: 27
	Files with the missing 2 AM entries: 1
	File Paths of the files with the duplicate 2 AM entries:
		C:\pyt\tx-soil-moisture\datasets\TxSON_data_2026-02-24\CB26.dat
	Files with the d

NOTE: above misses duplicate 1am with the same measurements.

In [18]:
# soil_files_dict = read_all_files_into_dict(soil_paths, read_soil)
# met_files_dict = read_all_files_into_dict(met_paths, read_met)

for year in range(2014, 2025):
    print_dst_full_summary(soil_files_dict, met_files_dict, year, diff_measurements=True)


Checking for DST indicators in 2014
	No DST indicators found in 2014.

Checking for DST indicators in 2015
On the day of the DST transitions:
	Files with the missing 2 AM entries: 0
	Files with the duplicate 1 AM entries: 0
One day AFTER the day of the DST transitions:
	Files with the duplicate 1 AM entries: 22
	Files with the missing 2 AM entries: 0

Checking for DST indicators in 2016
On the day of the DST transitions:
	Files with the missing 2 AM entries: 0
	Files with the duplicate 1 AM entries: 0
One day AFTER the day of the DST transitions:
	Files with the duplicate 1 AM entries: 25
	Files with the missing 2 AM entries: 1
	File Paths of the files with the duplicate 2 AM entries:
		C:\pyt\tx-soil-moisture\datasets\TxSON_data_2026-02-24\CB26.dat
	Files with the duplicate 1 AM entries AND the missing 2 AM entries: None

Checking for DST indicators in 2017
	No DST indicators found in 2017.

Checking for DST indicators in 2018
	No DST indicators found in 2018.

Checking for DST indic

The difference between with and without different measurements is in 2016 and 2017.

Lets manually check:

In [15]:
def manual_soil_check(abs_data_path, year, day_offset = 0):
    soil = read_soil(abs_data_path)
    start_data, end_data = transition_day_data(year, soil, day_offset = day_offset)
    print("Data for the day DST starts:")
    print(start_data)
    print("\nDoes the data have a missing 2 AM entry on the DST start day?", missing_two_am_DST(soil, year, day_offset = day_offset))
    print("\nData for the day DST ends:")
    print(end_data)
    print("\nDoes the data have a duplicate 1 AM entry on the DST end day?", duplicate_one_am_DST(soil, year, day_offset = day_offset))

def manual_met_check(abs_data_path, year, day_offset = 0):
    met = read_met(abs_data_path)
    start_data, end_data = transition_day_data(year, met, day_offset = day_offset)
    print("Data for the day DST starts:")
    print(start_data)
    print("\nDoes the data have a missing 2 AM entry on the DST start day?", missing_two_am_DST(met, year, day_offset = day_offset))
    print("\nData for the day DST ends:")
    print(end_data)
    print("\nDoes the data have a duplicate 1 AM entry on the DST end day?", duplicate_one_am_DST(met, year, day_offset = day_offset))


In [18]:
day_offset = 0
year = 2019

abs_soil_path = r'C:\pyt\tx-soil-moisture\datasets\TxSON_data_2026-02-24\CB25.dat'

manual_soil_check(abs_soil_path, year, day_offset = day_offset)
#manual_met_check(abs_met_path, year, day_offset = day_offset)

Data for the day DST starts:
                      Ppt  SWC_5 SWC_10 SWC_20 SWC_50    T_5   T_10   T_20  \
Date                                                                         
2019-03-10 00:00:00  0.00  0.282  0.333  0.365  0.425  16.34  17.65  17.40   
2019-03-10 01:00:00  0.00  0.281  0.332  0.364  0.424  15.80  17.07  17.20   
2019-03-10 02:00:00  0.00  0.280  0.331  0.364  0.425  15.41  16.61  17.00   
2019-03-10 03:00:00  0.00  0.279  0.330  0.363  0.425  15.05  16.24  16.79   
2019-03-10 04:00:00  0.00  0.278  0.329  0.363  0.425  14.70  15.90  16.59   

                      T_50  
Date                        
2019-03-10 00:00:00  14.61  
2019-03-10 01:00:00  14.68  
2019-03-10 02:00:00  14.74  
2019-03-10 03:00:00  14.79  
2019-03-10 04:00:00  14.84  

Does the data have a missing 2 AM entry on the DST start day? False

Data for the day DST ends:
                      Ppt  SWC_5 SWC_10 SWC_20 SWC_50    T_5   T_10   T_20  \
Date                                          

# Conclusion

**I cannot conclude that the network observes DST**. Out of all stations, there is not one that has indication of daylight savings time (DST) starting AND ending.

Criteria for indication of DST starting:
- A missing 2 am timestamp with a 1 am and 3 am timestamp surrounding it.

Criteria for indication of DST ending:
- A duplicate 1 am timestamp with or without the same measurements 

Findings:

- Insisting the duplicate 1am time stamps must have different measurements:

    - The dst indicators occurring ON the respective transition days: 
    
        - 2019: 2 files with duplicate one am timestamp. 

    - The dst indicators occurring one day AFTER the transition days where found in: 

        - 2015: 22 files with duplicate one am

        - 2016: 25 files with duplicate one am, 1 missing two am

        - 2019: 2 files with duplicate one am
